In [ ]:
!pip install -q \
    transformers \
    peft \
    trl \
    datasets \
    faiss-cpu \
    sentence-transformers \
    tqdm \
    torch \
    bitsandbytes \
    accelerate \
    paddleocr \
    pdf2image \
    paddlepaddle \
    pillow \
    pyngrok \
    nest_asyncio \
    uvicorn \
    fastapi \
    python-multipart

!apt-get update && apt-get install -y poppler-utils


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%%writefile utils.py
import os
import torch
import numpy as np
import faiss
import re
import gc

def reset_memory():
    gc.collect()

def clean_text(t):
    """Clean and normalize the input text."""
    return re.sub(r'\s+', ' ', t.strip())

def save(embs, docs, embedding_path, document_path):
    """Save embeddings and documents to disk."""
    np.save(embedding_path, embs.cpu().numpy())
    with open(document_path, "w", encoding="utf-8") as f:
        f.writelines(f"{doc}\n" for doc in docs)

def load(embedding_path, document_path):
    """Load embeddings and documents from disk."""
    if not os.path.exists(embedding_path) or not os.path.exists(document_path):
        return None, None
    embs = torch.tensor(np.load(embedding_path))
    with open(document_path, "r", encoding="utf-8") as f:
        docs = f.read().splitlines()
    return embs, docs

def save_index(index, faiss_index_path):
    """Save FAISS index to disk."""
    faiss.write_index(index, faiss_index_path)

def load_index(faiss_index_path):
    """Load FAISS index from disk."""
    return faiss.read_index(faiss_index_path)

def build_index(embs):
    """Build a FAISS index from embeddings."""
    embs = embs.cpu().numpy().astype("float32")
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index

def clean_and_overwrite_answer_file(file_path):
    """Clean and format the answers in the provided file."""
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    qa_blocks = re.findall(r"Query: (.*?)\n+Answer: (.*?)(?=\n+Query:|\Z)", content, re.DOTALL)
    cleaned_output = ""
    for query, answer in qa_blocks:
        cleaned_output += f"Question: {query.strip()}\nAnswer: {answer.strip()}\n\n"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cleaned_output.strip())
    print(f"Answers cleaned and saved to: {file_path}")

def load_ocr_docs(ocr_file):
    """Load and clean OCR documents from a file."""
    if os.path.exists(ocr_file):
        with open(ocr_file, "r", encoding="utf-8") as file:
            ocr_docs = file.readlines()
        return [clean_text(doc) for doc in ocr_docs]
    return []


In [ ]:
%%writefile models.py
import torch
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    AutoModelForCausalLM, AutoModelForSeq2SeqLM, BitsAndBytesConfig
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

def load_encoder(name, quantized=False):
    tokenizer = AutoTokenizer.from_pretrained(name)
    if quantized:
        model = AutoModel.from_pretrained(name, device_map="auto", quantization_config=bnb_config)
    else:
        model = AutoModel.from_pretrained(name).to(device)
    return tokenizer, model

def load_reranker(name, quantized=False):
    tokenizer = AutoTokenizer.from_pretrained(name)
    if quantized:
        model = AutoModelForSequenceClassification.from_pretrained(name, device_map="auto", quantization_config=bnb_config)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(name).to(device)
    return tokenizer, model

def load_summarizer(name, quantized=False):
    tokenizer = AutoTokenizer.from_pretrained(name)
    if quantized:
        model = AutoModelForSeq2SeqLM.from_pretrained(name, device_map="auto", quantization_config=bnb_config)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)
    return tokenizer, model

def load_generator(model_name, quantized=True):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    if quantized:
        model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=bnb_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    return tokenizer, model

def encode_query(query, tokenizer, model, max_query_length):
    inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=max_query_length).to(device)
    with torch.no_grad():
        embeddings = model.base_model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings

def rerank(query, candidates, tokenizer, model):
    inputs = [tokenizer(query, doc, return_tensors="pt", padding=True, truncation=True).to(device) for doc in candidates]
    scores = [model(**input).logits.softmax(dim=-1).max().item() for input in inputs]
    ranked = [doc for _, doc in sorted(zip(scores, candidates), reverse=True)]
    return ranked

def summarize(text, tokenizer, model, max_input_len=512, max_output_len=150):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input_len).to(device)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_output_len,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def generate_answer(query, wiki_context, ocr_context, tokenizer, model, max_new_tokens, temperature, top_p):
    input_text = f"Question: {query}\n"
    input_text += f"Most relevant information (OCR): {ocr_context}\n"
    input_text += f"Additional reference (Wikipedia): {wiki_context}\n"
    input_text += "Answer:"

    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(device)
    model.config.pad_token_id = model.config.eos_token_id
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[1].strip()


In [ ]:
%%writefile ocr_pipeline.py
import os
from pdf2image import convert_from_path
from paddleocr import PaddleOCR
import re

# === Init OCR model ===
ocr = PaddleOCR(use_angle_cls=True, lang='en')

# === PDF to Image ===
def pdf_to_images(pdf_path, dpi=300):
    return convert_from_path(pdf_path, dpi)

def save_images(pages, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for i, page in enumerate(pages):
        img_path = os.path.join(out_dir, f"page_{i}.jpg")
        page.save(img_path, 'JPEG')
        paths.append(img_path)
    return paths

# === OCR Text Extraction ===
def extract_text_with_paddleocr(img_path):
    result = ocr.ocr(img_path, cls=True)
    lines = [line[1][0] for line in result[0]]
    return "\n".join(lines)

# === Cleaning & Chunking ===
def clean_text(text):
    return re.sub(r'\s+', ' ', text.strip())

def chunk_text(text, max_words=300):
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

# === Full Pipeline ===
def process_uploaded_files(input_folder, output_txt, output_pages, chunk_size=300):
    all_text = []

    for filename in os.listdir(input_folder):
        filepath = os.path.join(input_folder, filename)
        print(f"\n Processing: {filepath}")

        if filename.lower().endswith(".pdf"):
            pages = pdf_to_images(filepath)
            img_dir = os.path.join(output_pages, os.path.splitext(filename)[0])
            image_paths = save_images(pages, out_dir=img_dir)
        elif filename.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths = [filepath]
        else:
            print(f"Skipping unsupported file: {filename}")
            continue

        for img_path in image_paths:
            text = extract_text_with_paddleocr(img_path)
            all_text.append(clean_text(text))

    full_text = "\n\n".join(all_text)
    chunks = chunk_text(full_text, max_words=chunk_size)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write("\n\n".join(chunks))

    print(f"\n OCR complete. Output saved to: {output_txt}")
    return chunks


In [ ]:
%%writefile main.py
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from pathlib import Path
import shutil
import os
import torch
from datasets import load_dataset
from tqdm import tqdm

from ocr_pipeline import process_uploaded_files
from utils import (
    clean_text, load, save, build_index, save_index, load_index,
    load_ocr_docs, clean_and_overwrite_answer_file, reset_memory
)
from models import (
    load_encoder, load_reranker, load_generator,
    encode_query, rerank, generate_answer, device
)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BASE_DIR = "./backend"

# File paths
documents_and_index = os.path.join(BASE_DIR, "documents_and_index")
embedding_path = os.path.join(documents_and_index, "embeddings.npy")
document_path = os.path.join(documents_and_index, "documents.txt")
faiss_index_path = os.path.join(documents_and_index, "faiss_index.index")
ocr_docs_path = os.path.join(documents_and_index, "ocr_docs.txt")
answer_path = os.path.join(BASE_DIR, "answer.txt")
input_folder = os.path.join(BASE_DIR, "uploaded_files")
output_pages = os.path.join(BASE_DIR, "output_pages")

# Ensure folders exist
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_pages, exist_ok=True)
os.makedirs(documents_and_index, exist_ok=True)

# Settings
top_k = 10
docs_to_embed = 5000
batch_size = 8
max_query_length = 256
max_new_tokens = 200
temperature = 0.7
top_p = 0.9

encoder_model_name = "BAAI/bge-base-en-v1.5"
reranker_model_name = "BAAI/bge-reranker-large"
generator_model_name = "deepcogito/cogito-v1-preview-llama-3B"
summarizer_model_name = "facebook/bart-large-cnn"

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Declare globals that will be initialized on startup
gen_tok = gen_model = None
enc_tok = enc_model = None
rr_tok = rr_model = None
sum_tok = sum_model = None
embs = None
docs = None
index = None
ocr_docs = []

@app.on_event("startup")
async def startup_event():
    global gen_tok, gen_model
    global enc_tok, enc_model
    global rr_tok, rr_model
    global sum_tok, sum_model
    global embs, docs, index
    global ocr_docs

    print("Loading models...")
    gen_tok, gen_model = load_generator(generator_model_name)
    enc_tok, enc_model = load_encoder(encoder_model_name)
    rr_tok, rr_model = load_reranker(reranker_model_name)
    sum_tok = AutoTokenizer.from_pretrained(summarizer_model_name)
    sum_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model_name).to(device)

    print("Loading embeddings and documents...")
    embs, docs = load(embedding_path, document_path)

    if embs is None or docs is None:
        print("Embeddings or docs missing, building from Wikipedia dataset...")
        wiki = load_dataset("wikipedia", "20220301.en", split=f"train[:{docs_to_embed}]", trust_remote_code=True)
        docs = [clean_text(ex["text"]) for ex in wiki]
        all_embs = []
        for i in tqdm(range(0, len(docs), batch_size), desc="Embedding batches"):
            batch = docs[i:i+batch_size]
            inputs = enc_tok(batch, return_tensors="pt", padding=True, truncation=True).to(device)
            with torch.no_grad():
                outputs = enc_model(**inputs)
                embs_batch = outputs.last_hidden_state.mean(dim=1).cpu()
                all_embs.append(embs_batch)
            reset_memory()
        embs = torch.cat(all_embs, dim=0)
        save(embs, docs, embedding_path, document_path)
    else:
        print(f"Loaded {len(docs)} documents and embeddings.")

    if os.path.exists(faiss_index_path):
        print("Loading FAISS index...")
        index = load_index(faiss_index_path)
    else:
        print("Building FAISS index...")
        index = build_index(embs)
        save_index(index, faiss_index_path)

    ocr_docs = load_ocr_docs(ocr_docs_path)
    print(f"OCR docs loaded: {len(ocr_docs)}")

# API models
class QueryRequest(BaseModel):
    query: str

@app.post("/upload/")
async def upload_file(file: UploadFile = File(...)):
    save_path = Path(input_folder) / file.filename
    with save_path.open("wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    chunks = process_uploaded_files(
        input_folder=input_folder,
        output_txt=ocr_docs_path,
        output_pages=output_pages,
        chunk_size=300
    )

    global ocr_docs
    ocr_docs = chunks  # Update global OCR docs cache

    return {"message": f"Processed {file.filename}", "chunks": len(chunks)}

@app.post("/query/")
def answer_query(req: QueryRequest):
    query = req.query

    global ocr_docs
    if not ocr_docs:
        ocr_docs = load_ocr_docs(ocr_docs_path)
    ocr_context = " ".join(ocr_docs) if ocr_docs else ""

    query_emb = encode_query(query, enc_tok, enc_model, max_query_length)
    _, top_idx = index.search(query_emb.cpu().numpy(), top_k)
    candidates = [docs[i] for i in top_idx[0]]
    reranked = rerank(query, candidates, rr_tok, rr_model)
    context = " ".join(reranked[:top_k])[:2048]

    sum_inputs = sum_tok(context, return_tensors="pt", max_length=1024, truncation=True).to(device)
    with torch.no_grad():
        summary_ids = sum_model.generate(
            sum_inputs["input_ids"],
            max_length=256,
            num_beams=4,
            early_stopping=True
        )
    summary = sum_tok.decode(summary_ids[0], skip_special_tokens=True)

    answer = generate_answer(query, summary, ocr_context, gen_tok, gen_model, max_new_tokens, temperature, top_p)

    with open(answer_path, "w", encoding="utf-8") as f:
        f.write(f"Query: {query}\n\nAnswer: {answer}\n\n")
    clean_and_overwrite_answer_file(answer_path)

    return JSONResponse(content={"answer": answer})

@app.get("/status/")
def get_status():
    return {"status": "ok"}


In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import os

nest_asyncio.apply()

NGROK_AUTH_TOKEN = "<INSERT_NGROK_TOKEN>"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

uvicorn.run("main:app", host="0.0.0.0", port=8000)